In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
from datetime import datetime
import traceback
import warnings
import pickle
import joblib
import logging
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score, f1_score, precision_score, recall_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch

warnings.filterwarnings('ignore')

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("sklearn_automl_evaluation.log")
    ]
)
logger = logging.getLogger()

def train_and_evaluate_sklearn_models(X_train_path, y_train_path, X_test_path, y_test_path, output_dir):
    """Entrena y evalúa modelos usando scikit-learn"""
    
    model_name = os.path.basename(X_train_path).replace("_X_train.parquet", "")
    print(f"\n{'='*50}\nProcesando: {model_name}\n{'='*50}")
    logger.info(f"Iniciando procesamiento para {model_name}")
    
    try:
        # Cargar datos
        print("Cargando datos...")
        X_train = pq.read_table(X_train_path).to_pandas()
        y_train = pq.read_table(y_train_path).to_pandas().squeeze()
        X_test = pq.read_table(X_test_path).to_pandas()
        y_test = pq.read_table(y_test_path).to_pandas().squeeze()
        
        # Información sobre el dataset
        print(f"Datos de entrenamiento: {X_train.shape}")
        print(f"Datos de prueba: {X_test.shape}")
        print(f"Distribución de clases (train): {y_train.value_counts().to_dict()}")
        
        # Verificar si hay NaN en los datos
        nan_count_train = X_train.isna().sum().sum()
        nan_count_test = X_test.isna().sum().sum()
        
        if nan_count_train > 0 or nan_count_test > 0:
            print(f"¡Advertencia! Valores NaN detectados: Train={nan_count_train}, Test={nan_count_test}")
            print("Reemplazando con la mediana...")
            
            # Calcular medianas para imputación
            median_values = X_train.median()
            X_train = X_train.fillna(median_values)
            X_test = X_test.fillna(median_values)
        
        # Selección de características
        print("Seleccionando características...")
        selector = SelectKBest(f_classif, k=min(100, X_train.shape[1]))
        X_train_selected = selector.fit_transform(X_train, y_train)
        X_test_selected = selector.transform(X_test)
        
        # Obtener índices de características seleccionadas
        selected_features = selector.get_support(indices=True)
        feature_names = X_train.columns[selected_features].tolist()
        
        print(f"Características seleccionadas: {len(feature_names)}")
        
        # Escalar características
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_selected)
        X_test_scaled = scaler.transform(X_test_selected)
        
        # Definir modelos para evaluar
        models = {
            'RandomForest': {
                'model': RandomForestClassifier(random_state=42, class_weight='balanced'),
                'params': {
                    'n_estimators': [100, 200],
                    'max_depth': [None, 10, 20],
                    'min_samples_split': [2, 5],
                    'min_samples_leaf': [1, 2],
                    'max_features': ['sqrt', 'log2']
                }
            },
            'GradientBoosting': {
                'model': GradientBoostingClassifier(random_state=42),
                'params': {
                    'n_estimators': [100, 200],
                    'learning_rate': [0.01, 0.1],
                    'max_depth': [3, 5],
                    'min_samples_split': [2, 5],
                    'min_samples_leaf': [1, 2]
                }
            },
            'LogisticRegression': {
                'model': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
                'params': {
                    'C': [0.01, 0.1, 1.0, 10.0],
                    'solver': ['liblinear', 'saga'],
                    'penalty': ['l1', 'l2']
                }
            }
        }
        
        # Crear directorios para guardar resultados
        os.makedirs(os.path.join(output_dir, "plots"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "models"), exist_ok=True)
        
        # Resultados de todos los modelos
        model_results = []
        best_score = -1
        best_model_name = None
        best_model = None
        
        # Entrenar y evaluar cada modelo
        for model_name, model_info in models.items():
            print(f"\nEntrenando {model_name}...")
            
            # Definir y ejecutar GridSearchCV
            grid_search = GridSearchCV(
                estimator=model_info['model'],
                param_grid=model_info['params'],
                scoring='f1_macro',
                cv=5,
                n_jobs=-1,
                verbose=1
            )
            
            grid_search.fit(X_train_scaled, y_train)
            
            # Obtener el mejor modelo
            best_estimator = grid_search.best_estimator_
            
            # Evaluar en conjunto de prueba
            y_pred = best_estimator.predict(X_test_scaled)
            
            # Calcular métricas
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
            recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
            f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
            
            # Calcular probabilidades (si el modelo lo soporta)
            if hasattr(best_estimator, "predict_proba"):
                y_proba = best_estimator.predict_proba(X_test_scaled)
                
                # Calcular AUC para cada clase
                auc_scores = {}
                for i, class_idx in enumerate(sorted(np.unique(y_test))):
                    try:
                        # Verificar que el índice existe en la matriz de probabilidades
                        if i < y_proba.shape[1]:
                            fpr, tpr, _ = roc_curve((y_test == class_idx).astype(int), y_proba[:, i])
                            auc_value = auc(fpr, tpr)
                            auc_scores[f"Clase_{class_idx}"] = auc_value
                    except Exception as e:
                        print(f"Error calculando AUC para clase {class_idx}: {str(e)}")
                        auc_scores[f"Clase_{class_idx}"] = np.nan
                
                # AUC promedio
                mean_auc = np.mean([v for v in auc_scores.values() if not np.isnan(v)])
            else:
                y_proba = None
                auc_scores = {f"Clase_{idx}": np.nan for idx in np.unique(y_test)}
                mean_auc = np.nan
            
            # Guardar resultados
            result = {
                'model_type': model_name,
                'best_params': grid_search.best_params_,
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'mean_auc': mean_auc,
                'auc_scores': auc_scores,
                'estimator': best_estimator
            }
            
            model_results.append(result)
            
            # Actualizar el mejor modelo
            if f1 > best_score:
                best_score = f1
                best_model_name = model_name
                best_model = result
            
            print(f"  {model_name} - F1: {f1:.4f}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")
        
        # Generar gráficos y guardar para el mejor modelo
        print(f"\nGenerando gráficos para el mejor modelo: {best_model_name}...")
        
        # Matriz de confusión
        plt.figure(figsize=(10, 8))
        cm = confusion_matrix(y_test, best_model['estimator'].predict(X_test_scaled))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=sorted(np.unique(y_test)),
                    yticklabels=sorted(np.unique(y_test)))
        plt.title(f'Matriz de Confusión - {model_name} ({best_model_name})')
        plt.xlabel('Predicción')
        plt.ylabel('Valor Real')
        
        conf_matrix_path = os.path.join(output_dir, "plots", f"{model_name}_{best_model_name}_cm.png")
        plt.savefig(conf_matrix_path)
        plt.close()
        
        # Curva ROC (si aplica)
        if hasattr(best_model['estimator'], "predict_proba"):
            plt.figure(figsize=(10, 8))
            plt.plot([0, 1], [0, 1], 'k--', lw=2)  # Línea diagonal
            
            best_proba = best_model['estimator'].predict_proba(X_test_scaled)
            
            for i, class_idx in enumerate(sorted(np.unique(y_test))):
                if i < best_proba.shape[1] and f"Clase_{class_idx}" in best_model['auc_scores']:
                    if not np.isnan(best_model['auc_scores'][f"Clase_{class_idx}"]):
                        fpr, tpr, _ = roc_curve((y_test == class_idx).astype(int), best_proba[:, i])
                        plt.plot(fpr, tpr, lw=2, 
                                label=f'Clase {class_idx} (AUC = {best_model["auc_scores"][f"Clase_{class_idx}"]:.2f})')
            
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('Tasa de Falsos Positivos')
            plt.ylabel('Tasa de Verdaderos Positivos')
            plt.title(f'Curva ROC - {model_name} ({best_model_name})')
            plt.legend(loc="lower right")
            
            roc_plot_path = os.path.join(output_dir, "plots", f"{model_name}_{best_model_name}_roc.png")
            plt.savefig(roc_plot_path)
            plt.close()
        else:
            roc_plot_path = None
        
        # Guardar el mejor modelo
        best_model_path = os.path.join(output_dir, "models", f"{model_name}_{best_model_name}.joblib")
        joblib.dump(best_model['estimator'], best_model_path)
        
        # Guardar información sobre las características seleccionadas
        feature_importance_df = pd.DataFrame({
            'feature': feature_names,
            'index': selected_features
        })
        
        if hasattr(best_model['estimator'], 'feature_importances_'):
            feature_importance_df['importance'] = best_model['estimator'].feature_importances_
            
            # Gráfico de importancia de características
            plt.figure(figsize=(12, 10))
            sorted_idx = np.argsort(best_model['estimator'].feature_importances_)[-30:]  # Top 30 características
            plt.barh(range(len(sorted_idx)), 
                    best_model['estimator'].feature_importances_[sorted_idx])
            plt.yticks(range(len(sorted_idx)), [feature_names[i] for i in sorted_idx])
            plt.title(f'Importancia de Características - {model_name} ({best_model_name})')
            
            feature_imp_path = os.path.join(output_dir, "plots", f"{model_name}_{best_model_name}_feat_imp.png")
            plt.tight_layout()
            plt.savefig(feature_imp_path)
            plt.close()
        else:
            feature_imp_path = None
        
        feature_importance_df.to_csv(os.path.join(output_dir, f"{model_name}_selected_features.csv"), index=False)
        
        # Guardar resultados comparativos
        results_df = pd.DataFrame([
            {
                'model_type': r['model_type'],
                'accuracy': r['accuracy'],
                'precision': r['precision'],
                'recall': r['recall'],
                'f1': r['f1'],
                'mean_auc': r['mean_auc']
            } for r in model_results
        ])
        
        results_df.to_csv(os.path.join(output_dir, f"{model_name}_model_comparison.csv"), index=False)
        
        # Variable para almacenar detalles del modelo
        final_result = {
            'model_name': f"{model_name}_{best_model_name}",
            'best_algorithm': best_model_name,
            'accuracy': best_model['accuracy'],
            'precision': best_model['precision'],
            'recall': best_model['recall'],
            'f1': best_model['f1'],
            'mean_auc': best_model['mean_auc'],
            'auc_scores': best_model['auc_scores'],
            'best_params': best_model['best_params'],
            'confusion_matrix_path': conf_matrix_path,
            'roc_curve_path': roc_plot_path,
            'feature_importance_path': feature_imp_path,
            'model_path': best_model_path,
            'feature_importance': feature_importance_df,
            'all_models_results': results_df.to_dict()
        }
        
        # Guardar detalles del modelo
        with open(os.path.join(output_dir, f"{model_name}_model_details.pkl"), 'wb') as f:
            pickle.dump(final_result, f)
        
        # Imprimir resultados
        print(f"\n✅ Evaluación de modelos completada para {model_name}:")
        print(f"   - Mejor algoritmo: {best_model_name}")
        print(f"   - Accuracy: {best_model['accuracy']:.4f}")
        print(f"   - Precision: {best_model['precision']:.4f}")
        print(f"   - Recall: {best_model['recall']:.4f}")
        print(f"   - F1 Score: {best_model['f1']:.4f}")
        print(f"   - AUC Promedio: {best_model['mean_auc']:.4f}")
        print(f"   - Mejores parámetros: {best_model['best_params']}")
        
        return final_result
        
    except Exception as e:
        print(f"\n❌ Error procesando {model_name}: {str(e)}")
        logger.error(f"Error procesando {model_name}: {str(e)}")
        traceback.print_exc()
        return None

def generate_performance_plots(all_results, output_dir):
    """Genera gráficos comparativos del rendimiento de los modelos"""
    
    # Crear un DataFrame con los resultados
    results_data = []
    for result in all_results:
        if result:
            results_data.append({
                'model_name': result['model_name'],
                'accuracy': result['accuracy'],
                'precision': result['precision'],
                'recall': result['recall'],
                'f1': result['f1'],
                'mean_auc': result['mean_auc'],
                'best_algorithm': result.get('best_algorithm', 'N/A')
            })
    
    if not results_data:
        print("No hay suficientes datos para generar gráficos.")
        return None
    
    results_df = pd.DataFrame(results_data)
    
    # Ordenar por F1 score
    results_df = results_df.sort_values('f1', ascending=False).reset_index(drop=True)
    
    # Gráfico de barras comparativo de F1 Score
    plt.figure(figsize=(12, 8))
    ax = sns.barplot(x='f1', y='model_name', data=results_df, palette='viridis')
    plt.title('Comparación de F1 Score entre Modelos', fontsize=14)
    plt.xlabel('F1 Score', fontsize=12)
    plt.ylabel('Modelo', fontsize=12)
    
    # Añadir valores en las barras
    for i, v in enumerate(results_df['f1']):
        ax.text(v + 0.01, i, f"{v:.4f}", va='center')
    
    f1_plot_path = os.path.join(output_dir, "plots", "f1_comparison.png")
    plt.tight_layout()
    plt.savefig(f1_plot_path)
    plt.close()
    
    # Gráfico de radar para métricas múltiples
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'mean_auc']
    
    # Tomar solo los 5 mejores modelos para el gráfico de radar
    top_models = results_df.head(5)
    
    # Crear el gráfico de radar
    plt.figure(figsize=(10, 10))
    
    # Número de variables
    N = len(metrics)
    
    # Ángulos para cada eje
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]  # Cerrar el polígono
    
    # Inicializar el gráfico de radar
    ax = plt.subplot(111, polar=True)
    
    # Añadir cada modelo
    for idx, row in top_models.iterrows():
        values = [row[metric] for metric in metrics]
        values += values[:1]  # Cerrar el polígono
        
        # Graficar
        ax.plot(angles, values, linewidth=2, label=row['model_name'])
        ax.fill(angles, values, alpha=0.1)
    
    # Establecer etiquetas
    plt.xticks(angles[:-1], metrics)
    
    # Añadir leyenda
    plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
    
    radar_plot_path = os.path.join(output_dir, "plots", "radar_metrics.png")
    plt.tight_layout()
    plt.savefig(radar_plot_path)
    plt.close()
    
    # Gráfico de algoritmos utilizados
    if 'best_algorithm' in results_df.columns:
        algorithm_counts = results_df['best_algorithm'].value_counts()
        
        plt.figure(figsize=(10, 6))
        algorithm_counts.plot(kind='bar', color=sns.color_palette('Set2'))
        plt.title('Distribución de Algoritmos Elegidos', fontsize=14)
        plt.xlabel('Algoritmo', fontsize=12)
        plt.ylabel('Frecuencia', fontsize=12)
        plt.xticks(rotation=45)
        
        algorithms_plot_path = os.path.join(output_dir, "plots", "algorithms_distribution.png")
        plt.tight_layout()
        plt.savefig(algorithms_plot_path)
        plt.close()
    
    return {
        'f1_plot': f1_plot_path,
        'radar_plot': radar_plot_path,
        'algorithms_plot': algorithms_plot_path if 'best_algorithm' in results_df.columns else None
    }

def generate_pdf_report(all_results, output_dir, plot_paths=None):
    """Genera un informe PDF con los resultados de todos los modelos"""
    
    if not all_results:
        print("No hay resultados para generar un informe.")
        return
    
    # Filtramos los resultados nulos
    filtered_results = [r for r in all_results if r]
    
    if not filtered_results:
        print("No hay resultados válidos para generar un informe.")
        return
    
    # Ordenar por F1 score
    sorted_results = sorted(filtered_results, key=lambda x: x['f1'], reverse=True)
    
    # Preparar PDF
    pdf_path = os.path.join(output_dir, f"informe_sklearn_automl_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf")
    doc = SimpleDocTemplate(pdf_path, pagesize=letter)
    elements = []
    
    # Estilos
    styles = getSampleStyleSheet()
    title_style = styles['Heading1']
    subtitle_style = styles['Heading2']
    normal_style = styles['Normal']
    
    # Título
    elements.append(Paragraph("Informe de Evaluación de Modelos con scikit-learn", title_style))
    elements.append(Spacer(1, 12))
    
    # Fecha
    elements.append(Paragraph(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M')}", normal_style))
    elements.append(Spacer(1, 12))
    
    # Resumen
    elements.append(Paragraph("Resumen de Resultados", subtitle_style))
    elements.append(Spacer(1, 6))
    
    # Tabla de resultados (Top 10)
    table_data = [["Modelo", "Algoritmo", "F1 Score", "Accuracy", "Precision", "Recall", "AUC"]]
    
    for idx, result in enumerate(sorted_results[:10]):  # Tomar solo los 10 mejores
        table_data.append([
            result['model_name'],
            result.get('best_algorithm', 'N/A'),
            f"{result['f1']:.4f}",
            f"{result['accuracy']:.4f}",
            f"{result['precision']:.4f}",
            f"{result['recall']:.4f}",
            f"{result['mean_auc']:.4f}"
        ])
    
    # Crear tabla
    table = Table(table_data, colWidths=[130, 100, 60, 60, 60, 60, 60])
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, 0), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
        ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
        ('GRID', (0, 0), (-1, -1), 1, colors.black),
        ('ALIGN', (1, 1), (-1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    ]))
    
    elements.append(table)
    elements.append(Spacer(1, 20))
    
    # Añadir gráficos si están disponibles
    if plot_paths:
        elements.append(Paragraph("Comparación de Modelos", subtitle_style))
        elements.append(Spacer(1, 10))
        
        if 'f1_plot' in plot_paths and plot_paths['f1_plot']:
            elements.append(Paragraph("Comparación de F1 Score", normal_style))
            elements.append(Spacer(1, 5))
            elements.append(Image(plot_paths['f1_plot'], width=6*inch, height=4*inch))
            elements.append(Spacer(1, 15))
        
        if 'radar_plot' in plot_paths and plot_paths['radar_plot']:
            elements.append(Paragraph("Métricas de los Mejores Modelos", normal_style))
            elements.append(Spacer(1, 5))
            elements.append(Image(plot_paths['radar_plot'], width=5*inch, height=5*inch))
            elements.append(Spacer(1, 15))
        
        if 'algorithms_plot' in plot_paths and plot_paths['algorithms_plot']:
            elements.append(Paragraph("Distribución de Algoritmos", normal_style))
            elements.append(Spacer(1, 5))
            elements.append(Image(plot_paths['algorithms_plot'], width=6*inch, height=3.5*inch))
            elements.append(Spacer(1, 15))
    
    # Mejor modelo
    best_model = sorted_results[0]
    elements.append(Paragraph("Mejor Modelo", subtitle_style))
    elements.append(Spacer(1, 6))
    elements.append(Paragraph(f"Nombre: {best_model['model_name']}", normal_style))
    elements.append(Paragraph(f"Algoritmo: {best_model.get('best_algorithm', 'N/A')}", normal_style))
    elements.append(Paragraph(f"F1 Score: {best_model['f1']:.4f}", normal_style))
    elements.append(Paragraph(f"Accuracy: {best_model['accuracy']:.4f}", normal_style))
    elements.append(Paragraph(f"Precision: {best_model['precision']:.4f}", normal_style))
    elements.append(Paragraph(f"Recall: {best_model['recall']:.4f}", normal_style))
    elements.append(Paragraph(f"AUC Promedio: {best_model['mean_auc']:.4f}", normal_style))
    
    # Parámetros del mejor modelo
    if 'best_params' in best_model:
        elements.append(Spacer(1, 10))
        elements.append(Paragraph("Parámetros del Mejor Modelo:", normal_style))
        param_text = str(best_model['best_params'])
        # Si es muy largo, truncarlo
        if len(param_text) > 1000:
            param_text = param_text[:1000] + "... [truncado]"
        elements.append(Paragraph(param_text, normal_style))
    
    elements.append(Spacer(1, 20))
    
    # Matriz de confusión del mejor modelo
    if 'confusion_matrix_path' in best_model and best_model['confusion_matrix_path']:
        elements.append(Paragraph("Matriz de Confusión - Mejor Modelo", subtitle_style))
        elements.append(Spacer(1, 5))
        elements.append(Image(best_model['confusion_matrix_path'], width=6*inch, height=6*inch))
        elements.append(Spacer(1, 15))
    
    # Curva ROC del mejor modelo
    if 'roc_curve_path' in best_model and best_model['roc_curve_path']:
        elements.append(Paragraph("Curva ROC - Mejor Modelo", subtitle_style))
        elements.append(Spacer(1, 5))
        elements.append(Image(best_model['roc_curve_path'], width=6*inch, height=6*inch))
        elements.append(Spacer(1, 15))
    
    # Importancia de características (si está disponible)
    if 'feature_importance_path' in best_model and best_model['feature_importance_path']:
        elements.append(Paragraph("Importancia de Características - Mejor Modelo", subtitle_style))
        elements.append(Spacer(1, 5))
        elements.append(Image(best_model['feature_importance_path'], width=6*inch, height=6*inch))
        elements.append(Spacer(1, 15))
    
    # Conclusiones
    elements.append(Paragraph("Conclusiones", subtitle_style))
    elements.append(Spacer(1, 6))
    
    # Tendencias en algoritmos
    if len(filtered_results) > 0:
        algorithm_counts = {}
        for result in filtered_results:
            alg = result.get('best_algorithm', 'N/A')
            algorithm_counts[alg] = algorithm_counts.get(alg, 0) + 1
        
        most_common_alg = max(algorithm_counts.items(), key=lambda x: x[1])[0]
        
        elements.append(Paragraph(f"El algoritmo más efectivo en general fue: {most_common_alg}", normal_style))
        elements.append(Spacer(1, 5))
    
    # Recomendaciones
    elements.append(Paragraph("Recomendaciones", subtitle_style))
    elements.append(Spacer(1, 6))
    elements.append(Paragraph("1. Utilizar el modelo con mejor F1 Score para un balance entre precisión y recall.", normal_style))
    elements.append(Paragraph("2. Para casos críticos donde se requiere alta sensibilidad, considerar modelos con mayor recall.", normal_style))
    elements.append(Paragraph("3. Evaluar posibles mejoras mediante ensamblaje de modelos o ajuste fino de hiperparámetros.", normal_style))
    elements.append(Paragraph("4. Monitorear el desempeño del modelo en producción y reentrenar periódicamente.", normal_style))
    elements.append(Paragraph("5. Para datasets desbalanceados, priorizar recall o precision según el contexto del problema.", normal_style))
    
    # Generate PDF
    doc.build(elements)
    
    print(f"\n✅ Informe PDF generado: {pdf_path}")
    
    return pdf_path

def main():
    """Función principal para ejecutar el pipeline completo"""
    
    # Definir directorio de entrada (archivos parquet)
    input_root = r"C:\Users\Administrador\Documents\PythonScripts\Tesis\tesisaustral\outputs\experiments\train_test_splits_no_smote_20250512_225619"
    
    # Directorio de salida para resultados
    output_root = os.path.join(input_root, "sklearn_automl_results")
    os.makedirs(output_root, exist_ok=True)
    os.makedirs(os.path.join(output_root, "plots"), exist_ok=True)
    
    logging.info(f"Iniciando evaluación de modelos. Directorio de entrada: {input_root}")
    logging.info(f"Directorio de salida: {output_root}")
    
    # Encontrar conjuntos de archivos train/test
    train_test_sets = []
    for root, _, files in os.walk(input_root):
        # Group files by prefix (e.g., "MaxAbs_all_features", "MaxAbs_Linear_selected")
        prefix_files = {}
        
        for file in files:
            if file.endswith(".parquet"):
                # Extract prefix (e.g., "MaxAbs_all_features")
                parts = file.split("_")
                
                # Assuming format like "MaxAbs_all_features_X_train.parquet"
                # We need to extract the prefix before "_X_train" or "_y_train", etc.
                if "X_train" in file or "y_train" in file or "X_test" in file or "y_test" in file:
                    # Find the index of "X_train", "y_train", etc.
                    for i, part in enumerate(parts):
                        if part in ["X", "y"]:
                            prefix = "_".join(parts[:i])
                            break
                    else:
                        # Skip if we can't determine the prefix
                        continue
                    
                    if prefix not in prefix_files:
                        prefix_files[prefix] = {}
                    
                    if "X_train" in file:
                        prefix_files[prefix]['X_train'] = os.path.join(root, file)
                    elif "y_train" in file:
                        prefix_files[prefix]['y_train'] = os.path.join(root, file)
                    elif "X_test" in file:
                        prefix_files[prefix]['X_test'] = os.path.join(root, file)
                    elif "y_test" in file:
                        prefix_files[prefix]['y_test'] = os.path.join(root, file)
        
        # Check which prefixes have complete sets
        for prefix, files_dict in prefix_files.items():
            if len(files_dict) == 4:  # All 4 files are present
                train_test_sets.append({
                    'prefix': prefix,
                    'files': files_dict
                })
    
    if not train_test_sets:
        logging.error("No se encontraron conjuntos completos de archivos de entrenamiento/prueba.")
        print("❌ No se encontraron conjuntos completos de archivos.")
        return
    
    logging.info(f"Encontrados {len(train_test_sets)} conjuntos de datos para evaluar.")
    print(f"\n🔍 Encontrados {len(train_test_sets)} conjuntos de datos para evaluar:")
    for i, train_set in enumerate(train_test_sets):
        print(f"  {i+1}. {train_set['prefix']}")
    
    # Evaluar cada conjunto de datos
    all_results = []
    for train_set in train_test_sets:
        try:
            result = train_and_evaluate_sklearn_models(
                train_set['files']['X_train'],
                train_set['files']['y_train'],
                train_set['files']['X_test'],
                train_set['files']['y_test'],
                output_root
            )
            all_results.append(result)
        except Exception as e:
            logging.error(f"Error evaluando {train_set['prefix']}: {str(e)}")
            print(f"❌ Error evaluando {train_set['prefix']}: {str(e)}")
    
    # Generar gráficos comparativos
    if len([r for r in all_results if r]) > 0:
        print("\n📊 Generando gráficos comparativos...")
        plot_paths = generate_performance_plots(all_results, output_root)
        
        # Generar informe PDF
        print("\n📝 Generando informe PDF...")
        pdf_path = generate_pdf_report(all_results, output_root, plot_paths)
        
        if pdf_path:
            print(f"\n✅ Proceso completado. Informe disponible en: {pdf_path}")
        else:
            print("\n⚠️ No se pudo generar el informe PDF.")
    else:
        logging.warning("No hay resultados suficientes para generar gráficos comparativos e informe.")
        print("\n⚠️ No hay resultados suficientes para generar gráficos comparativos e informe.")
    
    logging.info("Proceso de evaluación de modelos finalizado.")

if __name__ == "__main__":
    main()

2025-05-12 23:54:00,108 - INFO - Iniciando evaluación de modelos. Directorio de entrada: C:\Users\Administrador\Documents\PythonScripts\Tesis\tesisaustral\outputs\experiments\train_test_splits_no_smote_20250512_225619
2025-05-12 23:54:00,109 - INFO - Directorio de salida: C:\Users\Administrador\Documents\PythonScripts\Tesis\tesisaustral\outputs\experiments\train_test_splits_no_smote_20250512_225619\sklearn_automl_results
2025-05-12 23:54:00,140 - INFO - Encontrados 20 conjuntos de datos para evaluar.
2025-05-12 23:54:00,141 - INFO - Iniciando procesamiento para MaxAbs_all_features



🔍 Encontrados 20 conjuntos de datos para evaluar:
  1. MaxAbs_all_features
  2. MaxAbs_Linear_selected
  3. MaxAbs_Model-Based_selected
  4. MaxAbs_Nonlinear_selected
  5. MinMax_all_features
  6. MinMax_Linear_selected
  7. MinMax_Model-Based_selected
  8. MinMax_Nonlinear_selected
  9. NoNorm_all_features
  10. NoNorm_Linear_selected
  11. NoNorm_Model-Based_selected
  12. NoNorm_Nonlinear_selected
  13. Robust_all_features
  14. Robust_Linear_selected
  15. Robust_Model-Based_selected
  16. Robust_Nonlinear_selected
  17. Standard_all_features
  18. Standard_Linear_selected
  19. Standard_Model-Based_selected
  20. Standard_Nonlinear_selected

Procesando: MaxAbs_all_features
Cargando datos...
Datos de entrenamiento: (448388, 712)
Datos de prueba: (112098, 712)
Distribución de clases (train): {5: 124361, 4: 100161, 3: 95370, 2: 86994, 1: 41502}
Seleccionando características...
Características seleccionadas: 100

Entrenando RandomForest...
Fitting 5 folds for each of 48 candidates, t